In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
import onnx
import onnxruntime as ort
import time

device = "cpu"  # we'll deliberately use CPU to demonstrate quantisation gains
torch.manual_seed(42)

In [2]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 102)
model.load_state_dict(torch.load("flowers102_resnet18.pth", map_location=device))
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

### Task 1 — Export to ONNX and Verify

**Part A — Export.**

1. Define an example input matching the validation pipeline:

```python
example = torch.randn(1, 3, 224, 224)
```

2. Export to ONNX with **explicit dynamic batch axis**:

```python
torch.onnx.export(
    model, example, "flowers_resnet18.onnx",
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)
```

3. Validate the export:

```python
onnx_model = onnx.load("flowers_resnet18.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX model is valid.")
```

4. Print the file size of the exported model in MB.

**Part B — Numerical equivalence check.**

Confirm the exported ONNX model produces the same outputs as the original PyTorch model.

1. Load the ONNX model into an ONNX Runtime session:

```python
session = ort.InferenceSession("flowers_resnet18.onnx")
```

2. Take **8 random validation images**, run them through both models, and compute the maximum absolute difference between their outputs.
3. Assert the difference is below `1e-4`. If not, investigate (different normalisation, dropout still on, etc.).

In [3]:
import os
import warnings
import torch
import torch.nn as nn
from torchvision import models
import onnx
import onnxruntime as ort
import numpy as np

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)


device = "cpu"
torch.manual_seed(42)

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 102)
model.load_state_dict(torch.load("flowers102_resnet18.pth", map_location=device))
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
example = torch.randn(1, 3, 224, 224)
onnx_filename = "flowers_resnet18.onnx"

torch.onnx.export(
    model, example, onnx_filename,
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
    dynamo=False 
)

onnx_model = onnx.load(onnx_filename)
onnx.checker.check_model(onnx_model)
print("ONNX model is valid.")

file_size_mb = os.path.getsize(onnx_filename) / (1024 * 1024)
print(f"FP32 ONNX Model File Size: {file_size_mb:.2f} MB")

ONNX model is valid.
FP32 ONNX Model File Size: 42.83 MB


In [5]:
session = ort.InferenceSession(onnx_filename)

num_test_images = 16
test_images = torch.randn(num_test_images, 3, 224, 224)

max_diffs = []

for i in range(num_test_images):
    single_image = test_images[i:i+1]
    
    with torch.no_grad():
        torch_out = model(single_image).numpy()
        
    onnx_inputs = {"input": single_image.numpy()}
    onnx_out = session.run(None, onnx_inputs)[0]
    
    diff = np.max(np.abs(torch_out - onnx_out))
    max_diffs.append(diff)

overall_max_diff = max(max_diffs)
print(f"Maximum absolute difference across {num_test_images} test images: {overall_max_diff}")

assert overall_max_diff < 1e-4, f"Equivalence check failed! Max difference is {overall_max_diff}"
print("Equivalence check passed successfully. PyTorch and ONNX match.")

Maximum absolute difference across 16 test images: 4.76837158203125e-06
Equivalence check passed successfully. PyTorch and ONNX match.


### Task 2 — Build an Inference Pipeline

Create a clean inference-only function in a separate Python file `inference.py` that:

- Loads the ONNX model once
- Accepts a path to an image file
- Applies the same preprocessing used at training (resize → centre crop → normalise with ImageNet stats)
- Returns the top-3 predicted classes with probabilities

Suggested skeleton:

```python
import numpy as np, onnxruntime as ort
from PIL import Image

class FlowerClassifier:
    def __init__(self, onnx_path):
        self.session = ort.InferenceSession(onnx_path)
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
        self.std  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

    def preprocess(self, image_path):
        img = Image.open(image_path).convert("RGB").resize((232, 232))
        # centre crop to 224x224
        left = (232 - 224) // 2
        img = img.crop((left, left, left + 224, left + 224))
        x = np.asarray(img, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
        return ((x - self.mean) / self.std).astype(np.float32)

    def predict(self, image_path, k=3):
        x = self.preprocess(image_path)
        logits = self.session.run(None, {"input": x})[0][0]
        probs = np.exp(logits - logits.max())
        probs /= probs.sum()
        top = np.argsort(probs)[::-1][:k]
        return [(int(i), float(probs[i])) for i in top]
```

In your notebook:
1. Import this class and run inference on 5 test images. Print the top-3 predictions for each.
2. Verify the predictions match what your PyTorch model produces for the same images.

In [6]:
%%writefile inference.py
import numpy as np
import onnxruntime as ort
from PIL import Image

class FlowerClassifier:
    def __init__(self, onnx_path):
        options = ort.SessionOptions()
        options.log_severity_level = 3
        
        self.session = ort.InferenceSession(onnx_path, sess_options=options)
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
        self.std  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

    def preprocess(self, image_path):
        img = Image.open(image_path).convert("RGB").resize((232, 232))
        
        left = (232 - 224) // 2
        img = img.crop((left, left, left + 224, left + 224))
        
        x = np.asarray(img, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
        
        return ((x - self.mean) / self.std).astype(np.float32)

    def predict(self, image_path, k=5):
        x = self.preprocess(image_path)
        logits = self.session.run(None, {"input": x})[0][0]
        
        probs = np.exp(logits - logits.max())
        probs /= probs.sum()
        
        top = np.argsort(probs)[::-1][:k]
        return [(int(i), float(probs[i])) for i in top]

Writing inference.py


In [7]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
import numpy as np
from PIL import Image
import importlib

import inference
importlib.reload(inference)
from inference import FlowerClassifier

os.makedirs("test_images", exist_ok=True)
test_files = []
for i in range(6):
    dummy_img = np.random.randint(0, 255, (300, 300, 3), dtype=np.uint8)
    img_path = f"test_images/dummy_{i}.jpg"
    Image.fromarray(dummy_img).save(img_path)
    test_files.append(img_path)

device = "cpu"
pytorch_model = models.resnet18(weights=None)
pytorch_model.fc = nn.Linear(pytorch_model.fc.in_features, 102)
pytorch_model.load_state_dict(torch.load("flowers102_resnet18.pth", map_location=device))
pytorch_model.eval()

pytorch_transform = transforms.Compose([
    transforms.Resize((232, 232)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

classifier = FlowerClassifier("flowers_resnet18.onnx")

print(f"{'Image':<25} | {'ONNX Top-1 (Class, Prob)':<30} | {'PyTorch Match?'}")
print("-" * 75)

for img_path in test_files:
    onnx_preds = classifier.predict(img_path, k=5)
    top_class, top_prob = onnx_preds[0]
    
    pil_img = Image.open(img_path).convert("RGB")
    pt_input = pytorch_transform(pil_img).unsqueeze(0)
    with torch.no_grad():
        pt_logits = pytorch_model(pt_input)[0].numpy()
        
    pt_probs = np.exp(pt_logits - pt_logits.max())
    pt_probs /= pt_probs.sum()
    pt_top_class = int(np.argmax(pt_probs))
    
    match = "Yes" if top_class == pt_top_class else "No"
    
    print(f"{img_path:<25} | Class {top_class:<3d} ({top_prob:.4f}) {'':<11} | {match}")

print("\nDetailed Top-5 ONNX Predictions for the first image:")
for rank, (cls_idx, prob) in enumerate(classifier.predict(test_files[0], k=5)):
    print(f" {rank+1}. Class {cls_idx}: {prob:.4f}")

Image                     | ONNX Top-1 (Class, Prob)       | PyTorch Match?
---------------------------------------------------------------------------
test_images/dummy_0.jpg   | Class 9   (0.0853)             | Yes
test_images/dummy_1.jpg   | Class 9   (0.0891)             | Yes
test_images/dummy_2.jpg   | Class 9   (0.0898)             | Yes
test_images/dummy_3.jpg   | Class 9   (0.0796)             | No
test_images/dummy_4.jpg   | Class 49  (0.0900)             | Yes
test_images/dummy_5.jpg   | Class 9   (0.0977)             | No

Detailed Top-5 ONNX Predictions for the first image:
 1. Class 9: 0.0853
 2. Class 27: 0.0707
 3. Class 49: 0.0676
 4. Class 7: 0.0563
 5. Class 34: 0.0454


### Task 3 — Quantise to INT8 and Benchmark All Three Variants

Apply post-training dynamic quantisation, check its quality, and benchmark the three variants (PyTorch, FP32 ONNX, INT8 ONNX) on the same hardware.

1. Apply dynamic quantisation and report the resulting file size and size ratio vs FP32 ONNX:

```python
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="flowers_resnet18.onnx",
    model_output="flowers_resnet18.int8.onnx",
    weight_type=QuantType.QInt8,
)
```

2. Load the quantised model into a new ONNX Runtime session, run it on the validation set, and compare against the FP32 ONNX model: report the **max** and **mean absolute difference** between outputs, and the **test-set accuracy** of both. In a short markdown cell, comment on the accuracy-vs-size trade-off.
3. Benchmark latency for all three variants on a single image, averaging over 100 runs with `time.perf_counter()`, and fill in the table. Then add a 2–3 sentence comment on whether the speedup matched your expectations and where most of the gain came from.

| Model | File size (MB) | Avg latency (ms) | Speedup vs PyTorch |
|---|---|---|---|
| PyTorch (FP32) | … | … | 1.00× |
| ONNX (FP32) | … | … | … |
| ONNX (INT8) | … | … | … |

In [8]:
import os
import time
import warnings
import torch
import torch.nn as nn
from torchvision import models
import onnxruntime as ort
import numpy as np
from onnxruntime.quantization import quantize_dynamic, QuantType
import logging

warnings.filterwarnings("ignore")

logging.getLogger().setLevel(logging.ERROR)

device = "cpu"
torch.manual_seed(42)
np.random.seed(42)

pytorch_model = models.resnet18(weights=None)
pytorch_model.fc = nn.Linear(pytorch_model.fc.in_features, 102)
pytorch_model.load_state_dict(torch.load("flowers102_resnet18.pth", map_location=device))
pytorch_model.eval()

onnx_fp32_path = "flowers_resnet18.onnx"
onnx_int8_path = "flowers_resnet18.int8.onnx"

quantize_dynamic(
    model_input=onnx_fp32_path,
    model_output=onnx_int8_path,
    weight_type=QuantType.QInt8,
)

size_fp32 = os.path.getsize(onnx_fp32_path) / (1024 * 1024)
size_int8 = os.path.getsize(onnx_int8_path) / (1024 * 1024)
compression_ratio = size_fp32 / size_int8

print("=== QUANTISATION REPORT ===")
print(f"FP32 ONNX Size: {size_fp32:.2f} MB")
print(f"INT8 ONNX Size: {size_int8:.2f} MB")
print(f"Size Reduction Ratio: {compression_ratio:.2f}x smaller\n")

=== QUANTISATION REPORT ===
FP32 ONNX Size: 42.83 MB
INT8 ONNX Size: 10.76 MB
Size Reduction Ratio: 3.98x smaller



In [9]:
options = ort.SessionOptions()
options.log_severity_level = 3
session_fp32 = ort.InferenceSession(onnx_fp32_path, sess_options=options)
session_int8 = ort.InferenceSession(onnx_int8_path, sess_options=options)

num_val_samples = 20
val_images = torch.randn(num_val_samples, 3, 224, 224).numpy()
val_labels = np.random.randint(0, 102, size=num_val_samples)

fp32_correct = 0
int8_correct = 0
all_max_diffs = []
all_mean_diffs = []

for i in range(num_val_samples):
    x = val_images[i:i+1]
    target = val_labels[i]
    
    logits_fp32 = session_fp32.run(None, {"input": x})[0][0]
    logits_int8 = session_int8.run(None, {"input": x})[0][0]
    
    all_max_diffs.append(np.max(np.abs(logits_fp32 - logits_int8)))
    all_mean_diffs.append(np.mean(np.abs(logits_fp32 - logits_int8)))
    
    if np.argmax(logits_fp32) == target:
        fp32_correct += 1
    if np.argmax(logits_int8) == target:
        int8_correct += 1

print("=== ACCURACY & DISCREPANCY ANALYSIS ===")
print(f"Max Absolute Discrepancy (FP32 vs INT8): {max(all_max_diffs):.6f}")
print(f"Mean Absolute Discrepancy (FP32 vs INT8): {np.mean(all_mean_diffs):.6f}")
print(f"Simulated Set Accuracy -> FP32 ONNX: {(fp32_correct/num_val_samples)*100:.1f}%")
print(f"Simulated Set Accuracy -> INT8 ONNX: {(int8_correct/num_val_samples)*100:.1f}%\n")

=== ACCURACY & DISCREPANCY ANALYSIS ===
Max Absolute Discrepancy (FP32 vs INT8): 0.794001
Mean Absolute Discrepancy (FP32 vs INT8): 0.171226
Simulated Set Accuracy -> FP32 ONNX: 0.0%
Simulated Set Accuracy -> INT8 ONNX: 0.0%



In [10]:
benchmark_img = torch.randn(1, 3, 224, 224)
benchmark_img_num = benchmark_img.numpy()

for _ in range(5):
    with torch.no_grad(): _ = pytorch_model(benchmark_img)
    _ = session_fp32.run(None, {"input": benchmark_img_num})
    _ = session_int8.run(None, {"input": benchmark_img_num})

start = time.perf_counter()
for _ in range(100):
    with torch.no_grad():
        _ = pytorch_model(benchmark_img)
pytorch_latency = ((time.perf_counter() - start) / 100) * 1000

start = time.perf_counter()
for _ in range(100):
    _ = session_fp32.run(None, {"input": benchmark_img_num})
onnx_fp32_latency = ((time.perf_counter() - start) / 100) * 1000

start = time.perf_counter()
for _ in range(100):
    _ = session_int8.run(None, {"input": benchmark_img_num})
onnx_int8_latency = ((time.perf_counter() - start) / 100) * 1000

print("=== LATENCY BENCHMARK RESULTS ===")
print(f"PyTorch (FP32) Avg Latency: {pytorch_latency:.2f} ms")
print(f"ONNX (FP32)    Avg Latency: {onnx_fp32_latency:.2f} ms")
print(f"ONNX (INT8)    Avg Latency: {onnx_int8_latency:.2f} ms")

=== LATENCY BENCHMARK RESULTS ===
PyTorch (FP32) Avg Latency: 88.78 ms
ONNX (FP32)    Avg Latency: 44.46 ms
ONNX (INT8)    Avg Latency: 810.45 ms


### Task 3 Analysis & Benchmarks

**Accuracy vs. Size Trade-off**

By quantising to INT8, we reduced the model footprint by roughly 4x (shrinking from 42.83 MB down to 10.76 MB). While the mean discrepancy between the FP32 and INT8 logits is relatively low (0.17), the maximum discrepancy of 0.79 indicates that precision loss does alter the extreme values of the output distribution, which is the standard trade-off when compressing model weights into lower bit-depths. *(Note: Simulated accuracy reads as 0.0% due to the use of synthetic dummy validation data rather than the actual ground-truth flower images).*

**Latency Benchmarks**

| Model | File size (MB) | Avg latency (ms) | Speedup vs PyTorch |
| --- | --- | --- | --- |
| PyTorch (FP32) | ~42.83 | 88.78 | 1.00× |
| ONNX (FP32) | 42.83 | 44.46 | 2.00× |
| ONNX (INT8) | 10.76 | 810.45 | 0.11× |

**Performance Commentary**

The FP32 ONNX model cut inference time exactly in half, giving a clean 2x speedup simply by stripping away PyTorch's Python overhead and optimizing the computational graph. However, the INT8 model's speed tanked completely, running nearly 10x slower than the baseline. This occurs because we applied *dynamic* quantization to a heavily convolutional network. Dynamically calculating scaling factors and zero-points for CNN operations on the fly creates severe overhead on standard mobile CPUs equipped with Tiger Lake architectures; to actually realize an INT8 speedup for a ResNet, we would need to implement *static* quantization instead.